# Indonesian Dict Builder v2 — Tier-Filtered Baseline

Replacement for the original `dict_builder.ipynb`. Key differences:

1. **Tier-filtered pooling.** Only dicts classified `good` or `needs_attention`
   in `_quality_tiers.csv` contribute to the vocabulary. Broken dicts (which
   are dominated by OCR noise) are excluded.
2. **Three-resource pooling.** Indonesian tokens are collected from Billex,
   Morphology, AND Parcor (not just Parcor as in the original).
3. **Within-dict deduplication.** A word's per-dict contribution is binary —
   it either appears in that dict (1) or not (0). This prevents a single
   noisy dict from dominating frequencies via repetition.
4. **Tier weighting.** Each dict's contribution is multiplied by 2 if it's
   `good`, 1 if `needs_attention`. Cleaner dicts have more influence.
5. **Two strategies output in parallel:**
   - **Strategy B**: all `good` + `needs_attention` dicts, no cross-dict threshold
   - **Strategy C**: same as B, but only words appearing in ≥ `MIN_DICT_COUNT` dicts

## Outputs

- `indonesian_dict_strategy_B.json` — vocabulary + tier-weighted frequencies
- `indonesian_dict_strategy_C.json` — same, filtered to multi-dict words
- `_vocab_comparison.csv` — side-by-side stats for the two strategies
- `_vocab_provenance.csv` — per-word: which tier/resource/dicts it came from

The JSON files are drop-in replacements for the existing `indonesian_dict.json`
used by `PostProcessing pyspellchecker.ipynb`.

## 1. Configuration

In [1]:
import os
import re
import json
import pandas as pd
from pathlib import Path
from collections import Counter, defaultdict

# === Pooling options ===
MIN_DICT_COUNT_FOR_C = 3     # Strategy C cutoff
TIER_WEIGHTS = {
    "good": 2,
    "needs_attention": 1,
}
# `broken` and `not_applicable` are EXCLUDED automatically.

# === Paths ===
QUALITY_TIERS_PATH = Path("../evaluation/sanity_v2/_quality_tiers.csv")
LOOKUP_PATH        = Path("../Ekstraksi/12. Parallel Corpus - Spelling Checker/LookupIsFromIndonesia.csv")
BILLEX_DIR         = Path("../Ekstraksi/9. Bilingual Lexicon - Fixed")
MORPH_DIR          = Path("../Ekstraksi/10. Morphology - Fixed")
PARCOR_DIR         = Path("../Ekstraksi/11. Parallel Corpus - Fixed")

# Fallback to unfixed if fixed doesn't exist
for name, d, fallback in [
    ("BILLEX_DIR", BILLEX_DIR, Path("../Ekstraksi/9. Bilingual Lexicon")),
    ("MORPH_DIR",  MORPH_DIR,  Path("../Ekstraksi/10. Morphology")),
    ("PARCOR_DIR", PARCOR_DIR, Path("../Ekstraksi/11. Parallel Corpus")),
]:
    if not d.exists() and fallback.exists():
        print(f"  {name}: {d} not found, using {fallback}")
        if name == "BILLEX_DIR": BILLEX_DIR = fallback
        elif name == "MORPH_DIR": MORPH_DIR = fallback
        elif name == "PARCOR_DIR": PARCOR_DIR = fallback

DST_DIR = Path("../Code Tambahan")
DST_DIR.mkdir(parents=True, exist_ok=True)

print(f"Tier file:  {QUALITY_TIERS_PATH}  exists={QUALITY_TIERS_PATH.exists()}")
print(f"Lookup:     {LOOKUP_PATH}  exists={LOOKUP_PATH.exists()}")
print(f"Billex dir: {BILLEX_DIR}")
print(f"Morph dir:  {MORPH_DIR}")
print(f"Parcor dir: {PARCOR_DIR}")
print(f"Dest dir:   {DST_DIR.resolve()}")

assert QUALITY_TIERS_PATH.exists(), (
    "Run Distribution Sanity Checks v2.ipynb first to generate _quality_tiers.csv"
)

Tier file:  ..\evaluation\sanity_v2\_quality_tiers.csv  exists=True
Lookup:     ..\Ekstraksi\12. Parallel Corpus - Spelling Checker\LookupIsFromIndonesia.csv  exists=True
Billex dir: ..\Ekstraksi\9. Bilingual Lexicon - Fixed
Morph dir:  ..\Ekstraksi\10. Morphology - Fixed
Parcor dir: ..\Ekstraksi\11. Parallel Corpus - Fixed
Dest dir:   C:\Users\Legion\OneDrive\Documents\UNI\TA\tugas-akhir-data-mining\TAEkstraksiKamus\Code Tambahan


## 2. Load tier classifications

Read from `_quality_tiers.csv` produced by the sanity check v2. Only keep
dicts where `overall_tier_postfix` is `good` or `needs_attention`.

In [2]:
tiers_df = pd.read_csv(QUALITY_TIERS_PATH, dtype={"dict_id": str})
print(f"Loaded {len(tiers_df)} dictionary tiers")
print(f"\nDistribution of overall_tier_postfix:")
print(tiers_df["overall_tier_postfix"].value_counts())

# Filter to good + needs_attention
eligible_tiers = tiers_df[
    tiers_df["overall_tier_postfix"].isin(["good", "needs_attention"])
][["dict_id", "overall_tier_postfix"]].copy()

eligible_tiers["weight"] = eligible_tiers["overall_tier_postfix"].map(TIER_WEIGHTS)
print(f"\nEligible dictionaries: {len(eligible_tiers)}")
print(f"  good ({TIER_WEIGHTS['good']}×):            "
      f"{(eligible_tiers['overall_tier_postfix'] == 'good').sum()}")
print(f"  needs_attention ({TIER_WEIGHTS['needs_attention']}×): "
      f"{(eligible_tiers['overall_tier_postfix'] == 'needs_attention').sum()}")

DICT_WEIGHTS = dict(zip(eligible_tiers["dict_id"], eligible_tiers["weight"]))
DICT_TIERS   = dict(zip(eligible_tiers["dict_id"], eligible_tiers["overall_tier_postfix"]))

Loaded 68 dictionary tiers

Distribution of overall_tier_postfix:
overall_tier_postfix
broken             31
needs_attention    26
good               11
Name: count, dtype: int64

Eligible dictionaries: 37
  good (2×):            11
  needs_attention (1×): 26


## 3. Direction lookup

Load `LookupIsFromIndonesia.csv` to know which column is Indonesian in each
dictionary.

In [3]:
lookup_df = pd.read_csv(LOOKUP_PATH)

def parse_id(raw):
    m = re.search(r"\d+", str(raw))
    return m.group() if m else None

DIRECTION = {}
for _, row in lookup_df.iterrows():
    did = parse_id(row["kamus"])
    if did:
        DIRECTION[did] = int(row["is_from_indonesia"])

print(f"Loaded direction for {len(DIRECTION)} dictionaries")
print(f"  Indonesian → Regional (dir=1): {sum(v == 1 for v in DIRECTION.values())}")
print(f"  Regional → Indonesian (dir=0): {sum(v == 0 for v in DIRECTION.values())}")


def indonesian_cols_for(resource: str, direction: int) -> list[str]:
    """
    Given a resource type and direction, return which columns hold Indonesian
    content. For Morphology, the Indonesian side is whichever of `form` or
    the untouched `kata` column holds it.
    """
    if resource == "billex":
        return ["kata_asal"] if direction == 1 else ["kata_tujuan"]
    elif resource == "morph":
        # For Indonesian→Regional dicts, `kata` and `form` are Indonesian
        # For Regional→Indonesian dicts, neither column is guaranteed Indonesian
        # (the Indonesian glosses are embedded inside `form` before splitting)
        # Since we're using Fixed morphology, the `form` column is already split
        # and holds the source-side form (same language as `kata`).
        if direction == 1:
            return ["kata", "form"]  # both Indonesian
        else:
            return []  # Indonesian isn't reliably in any single column
    elif resource == "parcor":
        return ["kalimat_asal"] if direction == 1 else ["kalimat_tujuan"]
    return []

Loaded direction for 68 dictionaries
  Indonesian → Regional (dir=1): 28
  Regional → Indonesian (dir=0): 40


## 4. Extract Indonesian tokens per dict per resource

For each eligible dict, walk its Billex, Morph, and Parcor files and pull out
all Indonesian tokens. Within-dict deduplication: if a word appears 50 times
in Parcor, it counts as 1 for that dict.

Returns a map: `{dict_id: set(words)}` per resource.

In [4]:
TOKEN_RE = re.compile(r"[a-zA-Z]+")


def extract_tokens_from_csv(csv_path: Path, columns: list[str]) -> set:
    """Return the set of lowercase Indonesian tokens from the given columns."""
    tokens = set()
    if not columns:
        return tokens
    try:
        df = pd.read_csv(csv_path)
    except Exception as e:
        print(f"    failed to read {csv_path.name}: {e}")
        return tokens
    for col in columns:
        if col not in df.columns:
            continue
        for text in df[col].fillna("").astype(str):
            tokens.update(TOKEN_RE.findall(text.lower()))
    # Strip tokens that are pure noise: 1-char tokens and any with digits
    tokens = {t for t in tokens if len(t) >= 2 and t.isalpha()}
    return tokens


# Map: dict_id -> resource -> set of words
per_dict_resource_tokens = defaultdict(lambda: defaultdict(set))

eligible_ids = set(DICT_WEIGHTS.keys())
for resource, d in [
    ("billex", BILLEX_DIR),
    ("morph",  MORPH_DIR),
    ("parcor", PARCOR_DIR),
]:
    if not d.exists():
        print(f"  {resource}: dir not found, skipping")
        continue
    # Determine file suffix (Billex, Morphology, Parcor)
    suffix = {"billex": "Billex", "morph": "Morphology", "parcor": "Parcor"}[resource]
    files = sorted(d.glob(f"*_{suffix}.csv"))
    print(f"\n{resource}: scanning {len(files)} files")
    for f in files:
        did = parse_id(f.name)
        if did not in eligible_ids:
            continue
        direction = DIRECTION.get(did, 1)  # default to Ind->Reg
        cols = indonesian_cols_for(resource, direction)
        tokens = extract_tokens_from_csv(f, cols)
        per_dict_resource_tokens[did][resource] = tokens

print(f"\nExtracted tokens from {len(per_dict_resource_tokens)} dicts")


billex: scanning 68 files

morph: scanning 68 files

parcor: scanning 68 files

Extracted tokens from 37 dicts


## 5. Union per dict, then tier-weighted frequency

For each dict, union its three resource token sets — each word either
contributes or doesn't (deduplicated). Then accumulate a weighted frequency
across all eligible dicts.

In [5]:
# Per-word stats: weighted frequency, raw dict count, provenance
weighted_freq = Counter()           # token -> sum of weights
raw_dict_count = Counter()           # token -> number of distinct dicts
dicts_containing = defaultdict(set)  # token -> set of dict_ids
resource_contribution = defaultdict(set)  # token -> set of resource names

for did, per_resource in per_dict_resource_tokens.items():
    weight = DICT_WEIGHTS[did]
    # Union across resources for this dict
    all_tokens_this_dict = set()
    for resource, tokens in per_resource.items():
        all_tokens_this_dict |= tokens
        for t in tokens:
            resource_contribution[t].add(resource)

    for t in all_tokens_this_dict:
        weighted_freq[t] += weight
        raw_dict_count[t] += 1
        dicts_containing[t].add(did)

print(f"Total unique Indonesian tokens pooled: {len(weighted_freq)}")
print(f"Token appearing in ≥1 dict:            {len(weighted_freq)}")
print(f"Token appearing in ≥{MIN_DICT_COUNT_FOR_C} dicts: "
      f"{sum(1 for c in raw_dict_count.values() if c >= MIN_DICT_COUNT_FOR_C)}")

Total unique Indonesian tokens pooled: 140863
Token appearing in ≥1 dict:            140863
Token appearing in ≥3 dicts: 22601


## 6. Build the two strategy outputs

In [6]:
# Strategy B: all tokens with weighted_freq ≥ 1
strategy_B = {t: int(f) for t, f in weighted_freq.items()}

# Strategy C: only tokens in ≥ MIN_DICT_COUNT_FOR_C dicts
strategy_C = {
    t: int(weighted_freq[t])
    for t in weighted_freq
    if raw_dict_count[t] >= MIN_DICT_COUNT_FOR_C
}

print(f"Strategy B vocabulary size: {len(strategy_B):,}")
print(f"Strategy C vocabulary size: {len(strategy_C):,}")
print(f"  (reduction: {len(strategy_B) - len(strategy_C):,} tokens "
      f"= {(1 - len(strategy_C) / len(strategy_B)):.1%})")

# Save as JSON (compatible with pyspellchecker.word_frequency.load_dictionary)
path_B = DST_DIR / "indonesian_dict_strategy_B.json"
path_C = DST_DIR / "indonesian_dict_strategy_C.json"

with open(path_B, "w", encoding="utf-8") as f:
    json.dump(strategy_B, f)
with open(path_C, "w", encoding="utf-8") as f:
    json.dump(strategy_C, f)

print(f"\nWrote: {path_B}")
print(f"Wrote: {path_C}")

Strategy B vocabulary size: 140,863
Strategy C vocabulary size: 22,601
  (reduction: 118,262 tokens = 84.0%)

Wrote: ..\Code Tambahan\indonesian_dict_strategy_B.json
Wrote: ..\Code Tambahan\indonesian_dict_strategy_C.json


## 7. Comparison statistics

In [7]:
def strategy_stats(vocab: dict, name: str) -> dict:
    freqs = list(vocab.values())
    if not freqs:
        return {"strategy": name, "vocab_size": 0}
    freq_series = pd.Series(freqs)
    return {
        "strategy": name,
        "vocab_size": len(vocab),
        "total_weighted_tokens": sum(freqs),
        "freq_min": int(freq_series.min()),
        "freq_max": int(freq_series.max()),
        "freq_median": float(freq_series.median()),
        "freq_mean": round(float(freq_series.mean()), 2),
        "tokens_freq_1": int((freq_series == 1).sum()),
        "tokens_freq_2plus": int((freq_series >= 2).sum()),
        "tokens_freq_5plus": int((freq_series >= 5).sum()),
    }

comparison = pd.DataFrame([
    strategy_stats(strategy_B, "B_all_tiers"),
    strategy_stats(strategy_C, f"C_min{MIN_DICT_COUNT_FOR_C}_dicts"),
])
print(comparison.to_string(index=False))
comparison.to_csv(DST_DIR / "_vocab_comparison.csv", index=False)

    strategy  vocab_size  total_weighted_tokens  freq_min  freq_max  freq_median  freq_mean  tokens_freq_1  tokens_freq_2plus  tokens_freq_5plus
 B_all_tiers      140863                 441118         1        48          2.0       3.13          66029              74834              18339
C_min3_dicts       22601                 256712         3        48          8.0      11.36              0              22601              18339


## 8. Provenance export (for diagnostic / thesis appendix)

In [8]:
# Build a provenance table: for each token, which tier/resource/dicts contributed
provenance = []
for token in sorted(weighted_freq.keys()):
    dict_list = sorted(dicts_containing[token], key=lambda x: int(x))
    dict_tiers_for_token = [DICT_TIERS[did] for did in dict_list]
    in_strategy_C = raw_dict_count[token] >= MIN_DICT_COUNT_FOR_C
    provenance.append({
        "token": token,
        "weighted_freq": int(weighted_freq[token]),
        "n_dicts": raw_dict_count[token],
        "in_strategy_B": True,
        "in_strategy_C": in_strategy_C,
        "tiers": ";".join(dict_tiers_for_token),
        "resources": ";".join(sorted(resource_contribution[token])),
        "dict_ids": ";".join(dict_list),
    })

prov_df = pd.DataFrame(provenance)
prov_df.to_csv(DST_DIR / "_vocab_provenance.csv", index=False)
print(f"Wrote provenance: {len(prov_df):,} rows")

# Show a few samples
print("\n=== Sample high-frequency tokens (top 20 in Strategy C) ===")
top_C = prov_df[prov_df["in_strategy_C"]].nlargest(20, "weighted_freq")
print(top_C[["token", "weighted_freq", "n_dicts", "resources"]].to_string(index=False))

print("\n=== Sample low-frequency tokens (tokens that dropped from B to C) ===")
dropped = prov_df[(prov_df["in_strategy_B"]) & (~prov_df["in_strategy_C"])]
print(f"  Count: {len(dropped):,}")
print(dropped.sample(min(15, len(dropped)), random_state=42)[
    ["token", "weighted_freq", "n_dicts", "resources"]
].to_string(index=False))

Wrote provenance: 140,863 rows

=== Sample high-frequency tokens (top 20 in Strategy C) ===
   token  weighted_freq  n_dicts           resources
     ada             48       37 billex;morph;parcor
    adik             48       37 billex;morph;parcor
    agar             48       37 billex;morph;parcor
     air             48       37 billex;morph;parcor
    anak             48       37 billex;morph;parcor
   angin             48       37 billex;morph;parcor
     apa             48       37 billex;morph;parcor
     api             48       37 billex;morph;parcor
    ayam             48       37 billex;morph;parcor
    baik             48       37 billex;morph;parcor
    baju             48       37 billex;morph;parcor
  banyak             48       37 billex;morph;parcor
   bapak             48       37 billex;morph;parcor
  barang             48       37 billex;morph;parcor
    baru             48       37 billex;morph;parcor
    batu             48       37 billex;morph;parcor
   bawa

## 9. Sanity check against reference dicts

The 4 reference `good` dicts (#10, #35, #68, #91) should all have contributed
to the pool. Verify.

In [9]:
REFERENCE_DICTS = ["10", "35", "68", "91"]
print("Reference dict contributions:")
for did in REFERENCE_DICTS:
    if did not in per_dict_resource_tokens:
        print(f"  dict {did}: NOT ELIGIBLE (tier is not good/needs_attention)")
        continue
    per_res = per_dict_resource_tokens[did]
    total_tokens = len(set().union(*per_res.values())) if per_res else 0
    weight = DICT_WEIGHTS.get(did, 0)
    tier = DICT_TIERS.get(did, "???")
    print(f"  dict {did} (tier={tier}, weight={weight}×): "
          f"{total_tokens:,} unique Indonesian tokens contributed "
          f"(billex={len(per_res.get('billex', set())):,}, "
          f"morph={len(per_res.get('morph', set())):,}, "
          f"parcor={len(per_res.get('parcor', set())):,})")

Reference dict contributions:
  dict 10 (tier=good, weight=2×): 8,802 unique Indonesian tokens contributed (billex=3,783, morph=1,951, parcor=6,200)
  dict 35 (tier=good, weight=2×): 10,614 unique Indonesian tokens contributed (billex=1,943, morph=1,405, parcor=9,158)
  dict 68 (tier=good, weight=2×): 4,361 unique Indonesian tokens contributed (billex=2,642, morph=0, parcor=3,098)
  dict 91 (tier=good, weight=2×): 7,503 unique Indonesian tokens contributed (billex=5,344, morph=0, parcor=4,912)


## 10. Usage note

To use either strategy as a drop-in replacement for the previous
`indonesian_dict.json`, update the `PostProcessing pyspellchecker.ipynb`
notebook to point at the new path:

```python
DICT_PATH = '../Code Tambahan/indonesian_dict_strategy_C.json'
```

Recommended first comparison: run the pyspellchecker with Strategy B and
Strategy C on the same test dictionary and compare:
- How many tokens each flags as unknown (higher = more strict)
- How many it proposes corrections for
- A spot-check of the corrections — which ones look reasonable

Strategy B gives maximum coverage but may admit more noise. Strategy C is
stricter about what counts as a "real Indonesian word" and should produce
fewer false-positive corrections (where the spellchecker replaces a real
rare word with a more common but wrong alternative).